In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [74]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
import pandas as pd

books = pd.read_csv("books_cleaned.csv")

In [5]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883 A NOVEL THAT READERS and critics...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736 A memorable, mesmerizing heroine..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897 Lewis' work on the nature of lov...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222 On A Train Journey Home To North...
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014 This book tells the tale of a ma...
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...


In [6]:
books["tagged_description"]

0       9780002005883 A NOVEL THAT READERS and critics...
1       9780002261982 A new 'Christie for Christmas' -...
2       9780006178736 A memorable, mesmerizing heroine...
3       9780006280897 Lewis' work on the nature of lov...
4       9780006280934 "In The Problem of Pain, C.S. Le...
                              ...                        
5192    9788172235222 On A Train Journey Home To North...
5193    9788173031014 This book tells the tale of a ma...
5194    9788179921623 Wisdom to Create a Life of Passi...
5195    9788185300535 This collection of the timeless ...
5196    9789027712059 Since the three volume edition o...
Name: tagged_description, Length: 5197, dtype: object

In [64]:
books["tagged_description"].to_csv("tagged_description.txt",
                                   
                                   index = False,
                                   header = False)

In [65]:
raw_documents = TextLoader("tagged_description.txt", encoding="utf-8").load()
text_splitter = CharacterTextSplitter(
    chunk_size=2011,  # Must be > 0
    chunk_overlap=0, 
    separator="\n"
)
documents = text_splitter.split_documents(raw_documents)


Created a chunk of size 2012, which is longer than the specified 2011
Created a chunk of size 2834, which is longer than the specified 2011
Created a chunk of size 2510, which is longer than the specified 2011
Created a chunk of size 2285, which is longer than the specified 2011
Created a chunk of size 2616, which is longer than the specified 2011
Created a chunk of size 2032, which is longer than the specified 2011
Created a chunk of size 2762, which is longer than the specified 2011
Created a chunk of size 2141, which is longer than the specified 2011
Created a chunk of size 2209, which is longer than the specified 2011
Created a chunk of size 2877, which is longer than the specified 2011
Created a chunk of size 2400, which is longer than the specified 2011
Created a chunk of size 2122, which is longer than the specified 2011
Created a chunk of size 2181, which is longer than the specified 2011
Created a chunk of size 2171, which is longer than the specified 2011
Created a chunk of s

In [66]:
documents[0]

Document(metadata={'source': 'tagged_description.txt'}, page_content='"9780002005883 A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, G

In [75]:
db_books = Chroma.from_documents(
    documents,
    embedding=OpenAIEmbeddings())

In [76]:
query = "A book to teach children about nature"
docs = db_books.similarity_search(query, k = 10)
docs

[Document(id='4f5557fa-87d3-49c6-8f3d-e2dac92199bc', metadata={'source': 'tagged_description.txt'}, page_content='"9780067575208 First published more than three decades ago, this reissue of Rachel Carson\'s award-winning classic brings her unique vision to a new generation of readers. Stunning new photographs by Nick Kelsh beautifully complement Carson\'s intimate account of adventures with her young nephew, Roger, as they enjoy walks along the rocky coast of Maine and through dense forests and open fields, observing wildlife, strange plants, moonlight and storm clouds, and listening to the ""living music"" of insects in the underbrush. ""If a child is to keep alive his inborn sense of wonder."" Writes Carson, ""he needs the companionship of at least one adult who can share it, rediscovering with him the joy, excitement and mystery of the world we live in."" The Sense of Wonder is a refreshing antidote to indifference and a guide to capturing the simple power of discovery that Carson v

In [81]:
books[books["isbn13"] == int(docs[2].page_content.split()[0].strip().strip('"'))]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
3000,9780671631987,0671631985,Teach Your Child to Read in 100 Easy Lessons,Siegfried Engelmann;Phyllis Haddox;Elaine Bruner,Education,http://books.google.com/books/content?id=MUucJ...,"With more than half a million copies in print,...",1986.0,4.15,395.0,2083.0,Teach Your Child to Read in 100 Easy Lessons,9780671631987 With more than half a million co...


In [82]:
def retrieve_semantic_recommendations(
        query: str,
        top_k: int = 10,
) -> pd.DataFrame:
    recs = db_books.similarity_search(query, k = 50)

    books_list = []

    for i in range(0, len(recs)):
        books_list += [int(recs[i].page_content.strip('"').split()[0])]

    return books[books["isbn13"].isin(books_list)]

In [84]:
retrieve_semantic_recommendations("human relationships")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
84,9780029221303,0029221307,The Origins of the Civil Rights Movement,Aldon D. Morris,History,http://books.google.com/books/content?id=7vyHY...,A blending of scholarly research and interview...,1986.0,4.04,368.0,145.0,The Origins of the Civil Rights Movement,9780029221303 A blending of scholarly research...
101,9780060175641,0060175648,Identity,Milan Kundera,Fiction,http://books.google.com/books/content?id=D30Ex...,Milan Kundera's Identity translated from the F...,1998.0,3.68,176.0,260.0,Identity: A Novel,9780060175641 Milan Kundera's Identity transla...
156,9780060574215,0060574216,"Men Are from Mars, Women Are from Venus",John Gray,Family & Relationships,http://books.google.com/books/content?id=MUw_d...,Rediscover the most famous relationship book e...,2004.0,3.54,368.0,126108.0,"Men Are from Mars, Women Are from Venus: The C...",9780060574215 Rediscover the most famous relat...
189,9780060652890,0060652896,The Screwtape Letters,C. S. Lewis,Religion,http://books.google.com/books/content?id=v1kn7...,In this humorous and perceptive exchange betwe...,2001.0,4.22,224.0,116735.0,The Screwtape Letters,9780060652890 In this humorous and perceptive ...
284,9780060916091,0060916095,Think on These Things,Jiddu Krishnamurti,Religion,http://books.google.com/books/content?id=Qzzbm...,‘The material contained in this volume was ori...,1989.0,4.40,258.0,2288.0,Think on These Things,9780060916091 ‘The material contained in this ...
292,9780060925758,0060925752,Soul Mates,Thomas Moore,Psychology,http://books.google.com/books/content?id=7syEl...,This companion volume to Care of the Soul offe...,1994.0,4.00,288.0,4122.0,Soul Mates,9780060925758 This companion volume to Care of...
301,9780060930318,0060930314,Identity,Milan Kundera,Fiction,http://books.google.com/books/content?id=mXPU2...,There are situations in which we fail for a mo...,1999.0,3.68,168.0,13065.0,Identity: A Novel,9780060930318 There are situations in which we...
365,9780061122095,0061122092,By the River Piedra I Sat Down and Wept,Paulo Coelho,Fiction,http://books.google.com/books/content?id=9AHal...,"From Paulo Coelho, author of the international...",2006.0,3.57,208.0,68403.0,By the River Piedra I Sat Down and Wept: A Nov...,"9780061122095 From Paulo Coelho, author of the..."
370,9780061129735,0061129739,The Art of Loving,Erich Fromm,Self-Help,http://books.google.com/books/content?id=TRMED...,The fiftieth Anniversary Edition of the ground...,2006.0,4.03,192.0,35605.0,The Art of Loving,9780061129735 The fiftieth Anniversary Edition...
388,9780061196676,0061196673,Smithsonian Intimate Guide to Human Origins,Carl Zimmer,Social Science,http://books.google.com/books/content?id=xufuS...,From the savannas of Africa to modern-day labs...,2007.0,4.00,176.0,167.0,Smithsonian Intimate Guide to Human Origins,9780061196676 From the savannas of Africa to m...
